# 01 — Data Preprocessing & Split Audit
Audit of datasets, preprocessing, flow construction, and split protocol for VPN/non-VPN
classification across **ISCXVPN2016**, **USBVPN**, and **VNAT**.

**Integrity rule:** no numbers are invented. Any value that cannot be computed from project
files is emitted in a clearly marked **MISSING / NEEDS MANUAL INPUT** cell that names the
file/log/script required. All computed tables/figures are saved under `paper_audit_outputs/`.


In [1]:

# --- Setup: project root, imports, output folders, MISSING tracker ---
import os, sys, json, platform, warnings, hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid"); HAS_SNS = True
except Exception:
    HAS_SNS = False

# Locate project root (dir containing both 'artifacts' and 'data')
ROOT = Path.cwd().resolve()
while not ((ROOT / "artifacts").exists() and (ROOT / "data").exists()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("PROJECT ROOT:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT = ROOT / "paper_audit_outputs"
TBL, FIG, MET, LOG = OUT/"tables", OUT/"figures", OUT/"metrics", OUT/"logs"
for d in (TBL, FIG, MET, LOG):
    d.mkdir(parents=True, exist_ok=True)

MISSING = []  # list of dicts: section, what, needed
def mark_missing(section, what, needed):
    MISSING.append({"section": section, "what": what, "needed": needed})
    print(f"  [MISSING/{section}] {what}  -> needs: {needed}")

def save_table(df, name):
    p = TBL / name
    df.to_csv(p, index=False)
    print(f"  saved table: {p.relative_to(ROOT)}  ({df.shape[0]}x{df.shape[1]})")
    return p

def save_fig(fig, name):
    p = FIG / name
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved figure: {p.relative_to(ROOT)}")
    return p

def save_metric(obj, name):
    p = MET / name
    p.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print(f"  saved metric: {p.relative_to(ROOT)}")
    return p

GENERATED = []  # track generated files for final summary
print("Output folders ready under", OUT.relative_to(ROOT))


PROJECT ROOT: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
Output folders ready under paper_audit_outputs


## 1. Environment and project audit

In [2]:

# --- 1a. Environment versions ---
print("Python :", sys.version.replace("\n"," "))
print("OS     :", platform.platform())
print("CWD    :", Path.cwd())
print("ROOT   :", ROOT)
print()
def ver(mod):
    try:
        m = __import__(mod); return getattr(m, "__version__", "unknown")
    except Exception as e:
        return f"NOT INSTALLED ({e.__class__.__name__})"
env_rows = []
for mod in ["pandas","numpy","sklearn","xgboost","lightgbm","matplotlib",
            "seaborn","scipy","optuna","joblib"]:
    v = ver(mod); env_rows.append({"package": mod, "version": v}); print(f"  {mod:12s}: {v}")
env_df = pd.DataFrame(env_rows)
GENERATED.append(save_table(env_df, "audit_environment_versions.csv"))


Python : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
OS     : Windows-10-10.0.26200-SP0
CWD    : C:\Users\scoti\PycharmProjects\ai-vpn-firewall
ROOT   : C:\Users\scoti\PycharmProjects\ai-vpn-firewall

  pandas      : 3.0.0
  numpy       : 2.4.2
  sklearn     : 1.8.0


  xgboost     : 3.2.0
  lightgbm    : 4.6.0
  matplotlib  : 3.10.8
  seaborn     : 0.13.2
  scipy       : 1.17.0
  optuna      : 4.7.0
  joblib      : 1.5.3


  saved table: paper_audit_outputs\tables\audit_environment_versions.csv  (10x2)


In [3]:

# --- 1b. Recursive file discovery + role guessing ---
SKIP_DIRS = {".git", ".venv", "venv", "__pycache__", "node_modules", ".idea",
             ".ipynb_checkpoints", ".mypy_cache", ".pytest_cache"}

def guess_role(p: Path) -> str:
    s = str(p).replace("\\", "/").lower(); name = p.name.lower(); ext = p.suffix.lower()
    if "/data/raw/" in s: return "raw_dataset"
    if "/data/processed/" in s and ext == ".parquet" and "flow" in name: return "processed_flows"
    if "/data/processed/" in s and ext == ".parquet" and "feat" in name: return "feature_matrix"
    if "/data/splits/" in s or "split" in name and ext in (".txt",".json"): return "split_file"
    if name.endswith("_captures.txt"): return "split_capture_list"
    if "split" in name and ext == ".json": return "split_manifest"
    if ext in (".pkl",".joblib") or (ext==".pkl"): return "trained_model"
    if name.endswith("model.pkl") or "model" in name and ext in (".pkl",".joblib"): return "trained_model"
    if ext == ".parquet" and "flow" in name: return "processed_flows"
    if ext == ".parquet" and ("feat" in name or "feature" in name): return "feature_matrix"
    if ext == ".parquet": return "parquet_data"
    if ext == ".ipynb": return "notebook"
    if "/scripts/" in s and ext == ".py": return "script"
    if ("loader" in name or "parser" in name or "feature_extractor" in name
        or "feature_families" in name or "builder" in name): return "preprocessing_code"
    if ("train" in name) and ext == ".py": return "training_script"
    if name.endswith(".log") or "/logs/" in s or "_logs" in s: return "experiment_log"
    if "manifest" in name and ext == ".json": return "manifest"
    if ext == ".json": return "json_metadata"
    if ext == ".csv": return "csv_table"
    if "label" in name: return "labels"
    if "capture" in name: return "capture_identifiers"
    return "other"

RELEVANT_EXT = {".parquet",".pkl",".joblib",".txt",".json",".csv",".ipynb",".py",".log",".h5",".hdf5"}
rows = []
for dirpath, dirnames, filenames in os.walk(ROOT):
    dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
    for fn in filenames:
        p = Path(dirpath) / fn
        if p.suffix.lower() not in RELEVANT_EXT:
            continue
        try:
            st = p.stat()
        except OSError:
            continue
        role = guess_role(p)
        # keep only audit-relevant roles to bound table size
        if role in {"other","json_metadata","csv_table","parquet_data"} and "/data/" not in str(p).replace("\\","/").lower() \
           and "clean_pipeline" not in str(p).lower() and "unified_feature_contract" not in str(p).lower():
            continue
        rows.append({
            "path": str(p.relative_to(ROOT)),
            "file_name": p.name,
            "extension": p.suffix.lower(),
            "size_bytes": st.st_size,
            "last_modified": datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"),
            "guessed_role": role,
        })
found_df = pd.DataFrame(rows).sort_values(["guessed_role","path"]).reset_index(drop=True)
print(f"Discovered {len(found_df)} audit-relevant files")
print(found_df["guessed_role"].value_counts())
GENERATED.append(save_table(found_df, "audit_found_files.csv"))
found_df.head(20)


Discovered 1099 audit-relevant files
guessed_role
trained_model         447
json_metadata         202
csv_table              92
notebook               56
preprocessing_code     55
manifest               54
script                 51
raw_dataset            36
other                  26
experiment_log         20
feature_matrix         18
split_file             16
processed_flows         8
training_script         8
split_capture_list      6
parquet_data            4
Name: count, dtype: int64
  saved table: paper_audit_outputs\tables\audit_found_files.csv  (1099x6)


,path,file_name,extension,size_bytes,last_modified,guessed_role
0,artifacts\clean_pipeline\eval_v3\clean_lodo_re...,clean_lodo_results.csv,.csv,6177,2026-04-03T14:56:04,csv_table
1,artifacts\clean_pipeline\eval_v3\clean_policy_...,clean_policy_grid.csv,.csv,3206,2026-04-03T14:56:41,csv_table
2,artifacts\clean_pipeline\eval_v3\domain_weight...,domain_weighted_lodo.csv,.csv,223,2026-04-03T14:56:36,csv_table
3,artifacts\clean_pipeline\eval_v3\ensemble_mean...,ensemble_mean_results.csv,.csv,1213,2026-04-03T16:42:03,csv_table
4,artifacts\clean_pipeline\eval_v3\family_policy...,family_policy_grids.csv,.csv,23231,2026-04-03T17:54:55,csv_table
5,artifacts\clean_pipeline\eval_v3\family_search...,family_search_leaderboard.csv,.csv,8940,2026-04-03T17:58:41,csv_table
6,artifacts\clean_pipeline\eval_v3\family_search...,family_search_lodo_details.csv,.csv,8729,2026-04-03T17:54:41,csv_table
7,artifacts\clean_pipeline\eval_v3\family_search...,family_search_results.csv,.csv,8728,2026-04-03T17:54:41,csv_table
8,artifacts\clean_pipeline\eval_v3\feature_stabi...,feature_stability_rank.csv,.csv,5574,2026-04-03T16:42:18,csv_table
9,artifacts\clean_pipeline\eval_v3\final_candida...,final_candidate_table.csv,.csv,1074,2026-04-03T14:56:41,csv_table


## 2. Dataset composition audit

Canonical per-dataset source = the processed flow tables (one row per flow, with
`capture_id` and `label`):
`data/processed/{iscx,vnat,usbvpn}/flows.parquet`.


In [4]:

# --- 2a. Load per-dataset processed flows (canonical, has capture_id + label) ---
DATASET_FLOWS = {
    "iscx":   ROOT/"data"/"processed"/"iscx"/"flows.parquet",
    "usbvpn": ROOT/"data"/"processed"/"usbvpn"/"flows.parquet",
    "vnat":   ROOT/"data"/"processed"/"vnat"/"flows.parquet",
}
flows = {}
for ds, fp in DATASET_FLOWS.items():
    if fp.exists():
        df = pd.read_parquet(fp)
        if "label" not in df.columns:
            mark_missing("2", f"{ds}: no 'label' column in {fp.name}", str(fp.relative_to(ROOT)))
        if "capture_id" not in df.columns:
            mark_missing("2", f"{ds}: no 'capture_id' column in {fp.name}", str(fp.relative_to(ROOT)))
        flows[ds] = df
        print(f"{ds:7s}: {df.shape[0]:>7d} flows, {df['capture_id'].nunique() if 'capture_id' in df else '?'} captures, file={fp.name}")
    else:
        mark_missing("2", f"{ds}: processed flows.parquet not found", str(fp.relative_to(ROOT)))


iscx   :   76687 flows, 140 captures, file=flows.parquet
usbvpn :   58977 flows, 504 captures, file=flows.parquet
vnat   :   33711 flows, 165 captures, file=flows.parquet


In [5]:

# --- 2b. Composition + extended capture statistics ---
comp_rows, capstat_rows = [], []
for ds, df in flows.items():
    if "label" not in df or "capture_id" not in df:
        continue
    lab = df["label"].astype(int)
    total = len(df); vpn = int((lab==1).sum()); non = int((lab==0).sum())
    caps = df.groupby("capture_id")["label"].agg(["count","sum"])
    caps = caps.rename(columns={"count":"n","sum":"n_vpn"})
    caps["n_nonvpn"] = caps["n"] - caps["n_vpn"]
    only_vpn = int(((caps["n_vpn"]>0) & (caps["n_nonvpn"]==0)).sum())
    only_non = int(((caps["n_nonvpn"]>0) & (caps["n_vpn"]==0)).sum())
    both     = int(((caps["n_vpn"]>0) & (caps["n_nonvpn"]>0)).sum())
    comp_rows.append({
        "dataset": ds, "total_flows": total, "vpn_flows": vpn, "nonvpn_flows": non,
        "vpn_pct": round(100*vpn/total,4), "nonvpn_pct": round(100*non/total,4),
        "total_captures": int(len(caps)),
        "vpn_captures": int((caps["n_vpn"]>0).sum()),
        "nonvpn_captures": int((caps["n_nonvpn"]>0).sum()),
        "captures_only_vpn": only_vpn, "captures_only_nonvpn": only_non,
        "captures_both": both,
    })
    fpc = caps["n"].values
    capstat_rows.append({
        "dataset": ds,
        "flows_per_capture_min": int(np.min(fpc)),
        "flows_per_capture_Q1": float(np.percentile(fpc,25)),
        "flows_per_capture_median": float(np.percentile(fpc,50)),
        "flows_per_capture_Q3": float(np.percentile(fpc,75)),
        "flows_per_capture_max": int(np.max(fpc)),
        "vpn_flows_per_capture_median": float(np.percentile(caps["n_vpn"].values,50)),
        "nonvpn_flows_per_capture_median": float(np.percentile(caps["n_nonvpn"].values,50)),
    })
comp_df = pd.DataFrame(comp_rows); capstat_df = pd.DataFrame(capstat_rows)
display(comp_df); display(capstat_df)
GENERATED.append(save_table(comp_df, "dataset_composition.csv"))
GENERATED.append(save_table(capstat_df, "capture_statistics_extended.csv"))


,dataset,total_flows,vpn_flows,nonvpn_flows,vpn_pct,nonvpn_pct,total_captures,vpn_captures,nonvpn_captures,captures_only_vpn,captures_only_nonvpn,captures_both
0,iscx,76687,22433,54254,29.2527,70.7473,140,31,109,31,109,0
1,usbvpn,58977,8786,50191,14.8973,85.1027,504,23,504,0,481,23
2,vnat,33711,379,33332,1.1243,98.8757,165,82,83,82,83,0


,dataset,flows_per_capture_min,flows_per_capture_Q1,flows_per_capture_median,flows_per_capture_Q3,flows_per_capture_max,vpn_flows_per_capture_median,nonvpn_flows_per_capture_median
0,iscx,2,34.5,148.0,417.0,10504,0.0,52.0
1,usbvpn,7,100.0,100.0,100.0,700,0.0,100.0
2,vnat,1,1.0,5.0,26.0,11368,0.0,2.0


  saved table: paper_audit_outputs\tables\dataset_composition.csv  (3x12)
  saved table: paper_audit_outputs\tables\capture_statistics_extended.csv  (3x8)


In [6]:

# --- 2c. Plots: flows & captures per dataset (VPN/non-VPN), flows-per-capture distribution ---
if not comp_df.empty:
    ds_order = comp_df["dataset"].tolist()
    # flows per dataset
    fig, ax = plt.subplots(figsize=(7,4))
    x = np.arange(len(ds_order)); w=0.38
    ax.bar(x-w/2, comp_df["nonvpn_flows"], w, label="non-VPN", color="#4C72B0")
    ax.bar(x+w/2, comp_df["vpn_flows"], w, label="VPN", color="#C44E52")
    ax.set_xticks(x); ax.set_xticklabels(ds_order); ax.set_ylabel("flows"); ax.set_title("Flows per dataset by class")
    ax.legend(); GENERATED.append(save_fig(fig,"flows_per_dataset_by_class.png"))
    # captures per dataset
    fig, ax = plt.subplots(figsize=(7,4))
    ax.bar(x-w/2, comp_df["nonvpn_captures"], w, label="non-VPN captures", color="#4C72B0")
    ax.bar(x+w/2, comp_df["vpn_captures"], w, label="VPN captures", color="#C44E52")
    ax.set_xticks(x); ax.set_xticklabels(ds_order); ax.set_ylabel("captures"); ax.set_title("Captures per dataset by class")
    ax.legend(); GENERATED.append(save_fig(fig,"captures_per_dataset_by_class.png"))
    # flows-per-capture distribution (log scale)
    data = []
    labels = []
    for ds, df in flows.items():
        if "capture_id" in df:
            data.append(df.groupby("capture_id").size().values); labels.append(ds)
    if data:
        fig, ax = plt.subplots(figsize=(7,4))
        ax.boxplot(data, labels=labels, showfliers=True)
        ax.set_yscale("log"); ax.set_ylabel("flows per capture (log)"); ax.set_title("Flows per capture by dataset")
        GENERATED.append(save_fig(fig,"flows_per_capture_box.png"))


  saved figure: paper_audit_outputs\figures\flows_per_dataset_by_class.png


  saved figure: paper_audit_outputs\figures\captures_per_dataset_by_class.png


  saved figure: paper_audit_outputs\figures\flows_per_capture_box.png


## 3. Data cleaning and filtering audit

Per-flow removal/cleaning counters that the pipeline **logged** are read from the dataset
manifests and from per-flow quality columns (`packet_count`, `packet_count_full`,
`min_packets_ok` for ISCX/VNAT; `q_packet_count`, `q_min_packets_ok` for USBVPN).
Counters that were **not** logged are reconstructed from code behaviour where possible and
otherwise marked MISSING.


In [7]:

# --- 3a. Read manifests (raw vs kept) ---
manifest_paths = {
    "iscx_flows":   ROOT/"data"/"processed"/"iscx"/"flows_manifest.json",
    "iscx_feat":    ROOT/"data"/"processed"/"iscx"/"features_manifest.json",
    "vnat_flows":   ROOT/"data"/"processed"/"vnat"/"flows_manifest.json",
    "vnat_feat":    ROOT/"data"/"processed"/"vnat"/"features_manifest.json",
}
manifests = {}
for k, p in manifest_paths.items():
    if p.exists():
        try: manifests[k] = json.loads(p.read_text())
        except Exception as e: mark_missing("3", f"{k} manifest unreadable: {e}", str(p.relative_to(ROOT)))
    else:
        mark_missing("3", f"{k} manifest not found", str(p.relative_to(ROOT)))
if "usbvpn" not in str(list(manifest_paths.keys())):
    pass
usb_man = ROOT/"data"/"processed"/"usbvpn"/"flows_manifest.json"
if not usb_man.exists():
    mark_missing("3", "usbvpn flows_manifest.json not found (raw->kept counts unavailable)", str(usb_man.relative_to(ROOT)))
print("Loaded manifests:", list(manifests.keys()))


  [MISSING/3] usbvpn flows_manifest.json not found (raw->kept counts unavailable)  -> needs: data\processed\usbvpn\flows_manifest.json
Loaded manifests: ['iscx_flows', 'iscx_feat', 'vnat_flows', 'vnat_feat']


In [8]:

# --- 3b. Per-flow cleaning counters from quality columns ---
clean_rows = []
for ds, df in flows.items():
    n = len(df)
    pc_col   = "packet_count" if "packet_count" in df else ("q_packet_count" if "q_packet_count" in df else None)
    pcf_col  = "packet_count_full" if "packet_count_full" in df else None
    mpo_col  = "min_packets_ok" if "min_packets_ok" in df else ("q_min_packets_ok" if "q_min_packets_ok" in df else None)
    row = {"dataset": ds, "flows_in_processed_table": n}
    # too few packets (failed min_packets_ok)
    if mpo_col:
        bad = int((~df[mpo_col].astype(bool)).sum())
        row["flows_failing_min_packets_ok"] = bad
    else:
        row["flows_failing_min_packets_ok"] = "MISSING"
        mark_missing("3", f"{ds}: no min_packets_ok column", DATASET_FLOWS[ds].name)
    # window truncation: full packet count exceeded window (>300)
    if pcf_col and pc_col:
        trunc = int((df[pcf_col] > df[pc_col]).sum())
        row["flows_window_truncated_gt_window"] = trunc
    else:
        row["flows_window_truncated_gt_window"] = "MISSING (no packet_count_full)"
        mark_missing("3", f"{ds}: packet_count_full not stored; window-truncation count not reconstructable", DATASET_FLOWS[ds].name)
    # missing labels / capture_id
    row["missing_labels"] = int(df["label"].isna().sum()) if "label" in df else "MISSING"
    row["missing_capture_id"] = int(df["capture_id"].isna().sum()) if "capture_id" in df else "MISSING"
    # non-finite feature values across numeric cols
    num = df.select_dtypes(include=[np.number])
    nonfinite = int(np.isinf(num.values).sum()) if num.size else 0
    nan_vals  = int(num.isna().values.sum()) if num.size else 0
    row["nonfinite_numeric_cells"] = nonfinite
    row["nan_numeric_cells"] = nan_vals
    clean_rows.append(row)
clean_df = pd.DataFrame(clean_rows)
display(clean_df)
GENERATED.append(save_table(clean_df, "cleaning_summary.csv"))


  [MISSING/3] usbvpn: packet_count_full not stored; window-truncation count not reconstructable  -> needs: flows.parquet


,dataset,flows_in_processed_table,flows_failing_min_packets_ok,flows_window_truncated_gt_window,missing_labels,missing_capture_id,nonfinite_numeric_cells,nan_numeric_cells
0,iscx,76687,64886,1262,0,0,0,0
1,usbvpn,58977,6273,MISSING (no packet_count_full),0,0,0,0
2,vnat,33711,25604,935,0,0,0,0


  saved table: paper_audit_outputs\tables\cleaning_summary.csv  (3x8)


In [9]:

# --- 3c. Cleaning reason table: raw->kept from manifests where available + code-level reasons ---
reason_rows = []
def manifest_count(d, *keys):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return None
for ds in flows:
    fm = manifests.get(f"{ds}_flows", {})
    raw = manifest_count(fm, "n_raw_flows","raw_flows","n_input_flows","total_input")
    kept = manifest_count(fm, "n_flows","kept_flows","n_output_flows","total")
    reason_rows.append({
        "dataset": ds,
        "raw_flows_before_filter": raw if raw is not None else "MISSING",
        "flows_after_filter": kept if kept is not None else len(flows[ds]),
        "removed_flows": (raw-kept) if (raw is not None and kept is not None) else "MISSING",
        "removal_pct": round(100*(raw-kept)/raw,4) if (raw and kept is not None) else "MISSING",
    })
    if raw is None:
        mark_missing("3", f"{ds}: raw (pre-filter) flow count not in manifest", f"data/processed/{ds}/flows_manifest.json")
reason_df = pd.DataFrame(reason_rows)
display(reason_df)
GENERATED.append(save_table(reason_df, "cleaning_reason_table.csv"))

# code-level cleaning behaviours (facts from src/clean_pipeline/feature_extractor.py)
code_behaviours = pd.DataFrame([
    {"behaviour":"epsilon in rate/CV denominators","value":"_EPS = 1e-9 (feature_extractor.py:28)","per_flow_count":"N/A (constant)"},
    {"behaviour":"signed packet sizes -> absolute","value":"sz = np.abs(sizes[:n]) (L96)","per_flow_count":"NOT LOGGED"},
    {"behaviour":"timestamp sorting before IAT","value":"order = np.argsort(ts) (L100); _iat re-sorts (L62)","per_flow_count":"NOT LOGGED"},
    {"behaviour":"negative IAT clamped to 0","value":"np.maximum(diff,0.0) (L64)","per_flow_count":"NOT LOGGED"},
    {"behaviour":"empty IAT (1 packet) -> stats 0.0","value":"_safe_stats empty -> all 0.0 (L33)","per_flow_count":"NOT LOGGED"},
    {"behaviour":"packet-window truncation","value":"max_packets=300 (L76)","per_flow_count":"see cleaning_summary (where packet_count_full available)"},
    {"behaviour":"too-few-packets drop","value":"min_packets=3 (extract_features_batch L231/280)","per_flow_count":"see cleaning_summary.flows_failing_min_packets_ok"},
    {"behaviour":"non-finite features replaced","value":"to_numeric->fillna(0.0); replace(+/-inf,0.0) (L322,325)","per_flow_count":"NOT LOGGED per-flow"},
])
GENERATED.append(save_table(code_behaviours, "cleaning_code_behaviours.csv"))
mark_missing("3", "Per-flow counts for sign-flips, IAT clamps, empty-IAT, sort-affected, inf-replacements are NOT logged by the pipeline", "instrument src/clean_pipeline/feature_extractor.py to emit per-flow cleaning counters, or a cleaning log")
display(code_behaviours)


  [MISSING/3] iscx: raw (pre-filter) flow count not in manifest  -> needs: data/processed/iscx/flows_manifest.json
  [MISSING/3] usbvpn: raw (pre-filter) flow count not in manifest  -> needs: data/processed/usbvpn/flows_manifest.json
  [MISSING/3] vnat: raw (pre-filter) flow count not in manifest  -> needs: data/processed/vnat/flows_manifest.json


,dataset,raw_flows_before_filter,flows_after_filter,removed_flows,removal_pct
0,iscx,MISSING,76687,MISSING,MISSING
1,usbvpn,MISSING,58977,MISSING,MISSING
2,vnat,MISSING,33711,MISSING,MISSING


  saved table: paper_audit_outputs\tables\cleaning_reason_table.csv  (3x5)
  saved table: paper_audit_outputs\tables\cleaning_code_behaviours.csv  (8x3)
  [MISSING/3] Per-flow counts for sign-flips, IAT clamps, empty-IAT, sort-affected, inf-replacements are NOT logged by the pipeline  -> needs: instrument src/clean_pipeline/feature_extractor.py to emit per-flow cleaning counters, or a cleaning log


,behaviour,value,per_flow_count
0,epsilon in rate/CV denominators,_EPS = 1e-9 (feature_extractor.py:28),N/A (constant)
1,signed packet sizes -> absolute,sz = np.abs(sizes[:n]) (L96),NOT LOGGED
2,timestamp sorting before IAT,order = np.argsort(ts) (L100); _iat re-sorts (...,NOT LOGGED
3,negative IAT clamped to 0,"np.maximum(diff,0.0) (L64)",NOT LOGGED
4,empty IAT (1 packet) -> stats 0.0,_safe_stats empty -> all 0.0 (L33),NOT LOGGED
5,packet-window truncation,max_packets=300 (L76),see cleaning_summary (where packet_count_full ...
6,too-few-packets drop,min_packets=3 (extract_features_batch L231/280),see cleaning_summary.flows_failing_min_packets_ok
7,non-finite features replaced,"to_numeric->fillna(0.0); replace(+/-inf,0.0) (...",NOT LOGGED per-flow


In [10]:

# --- 3d. Removed-flows-by-reason plot (only reasons with real numeric counts) ---
plot_rows = []
for _, r in clean_df.iterrows():
    ds = r["dataset"]
    v = r.get("flows_failing_min_packets_ok")
    if isinstance(v,(int,np.integer)):
        plot_rows.append({"dataset":ds,"reason":"min_packets (<3)","count":int(v)})
    v = r.get("flows_window_truncated_gt_window")
    if isinstance(v,(int,np.integer)):
        plot_rows.append({"dataset":ds,"reason":"window_truncated (>300)","count":int(v)})
if plot_rows:
    pr = pd.DataFrame(plot_rows)
    piv = pr.pivot(index="dataset", columns="reason", values="count").fillna(0)
    fig, ax = plt.subplots(figsize=(7,4))
    piv.plot(kind="bar", ax=ax); ax.set_ylabel("flows"); ax.set_title("Removed/affected flows by reason (logged only)")
    GENERATED.append(save_fig(fig,"removed_flows_by_reason.png"))
else:
    print("No numerically-logged removal reasons available to plot (see MISSING markers).")


  saved figure: paper_audit_outputs\figures\removed_flows_by_reason.png

## 4. Flow-construction and loader audit

Facts are read from the loader/extractor source code. Each row is tagged
KNOWN / ASSUMPTION / UNKNOWN.


In [11]:

# --- 4. Flow construction audit from code ---
loader_files = {
    "iscx":   ROOT/"src"/"clean_pipeline"/"iscx_loader.py",
    "vnat":   ROOT/"src"/"clean_pipeline"/"vnat_loader.py",
    "usbvpn": ROOT/"src"/"clean_pipeline"/"usbvpn_parser.py",
    "extractor": ROOT/"src"/"clean_pipeline"/"feature_extractor.py",
    "builder_legacy": ROOT/"src"/"flow"/"builder.py",
}
for k,p in loader_files.items():
    if not p.exists():
        mark_missing("4", f"loader/code file missing: {k}", str(p.relative_to(ROOT)))

fc_rows = []
for ds in ["iscx","vnat","usbvpn"]:
    lf = loader_files[ds]
    fc_rows.append({
        "dataset": ds,
        "loader_file": str(lf.relative_to(ROOT)) if lf.exists() else "MISSING",
        "source_format": {"iscx":"pcap/pcapng (raw)","vnat":"pcap (raw)","usbvpn":"pcap (raw, per-protocol folders)"}[ds],
        "loader_status": "KNOWN" if lf.exists() else "UNKNOWN",
        "flows_native_or_reconstructed": "dataset-native segmentation via clean_pipeline loader (KNOWN)",
        "bidirectional": "yes (directions array 0/1) [KNOWN from extractor]",
        "direction_representation": "directions in {0,1}; fwd=1/bwd=0 [KNOWN extractor L106-107]",
        "packet_size_representation": "absolute bytes; np.abs() applied [KNOWN extractor L96]",
        "signed_to_abs": "yes (np.abs) [KNOWN]",
        "timestamps_sorted": "yes (np.argsort) [KNOWN extractor L100]",
        "packet_window_applied": "yes [KNOWN]",
        "packet_window_len": "max_packets=300 [KNOWN run_metadata/config]",
        "min_packets": "3 [KNOWN config]",
        "flow_timeout_if_reconstructed": "legacy builder inactivity_timeout=120s [KNOWN src/flow/builder.py]; clean loaders use dataset-native flows (ASSUMPTION: no re-timeout)",
        "active_timeout": "UNKNOWN (not specified in clean loaders)",
        "five_tuple_logic": "legacy builder: A=min(endpoint),B=max(endpoint),proto [KNOWN builder.py]; clean loader keying UNKNOWN per-dataset",
        "merge_policy": "UNKNOWN (whether builder produced current data/processed/* not confirmed)",
        "remaining_uncertainties": "exact clean-loader 5-tuple keying & whether legacy builder generated processed parquet",
    })
fc_df = pd.DataFrame(fc_rows)
display(fc_df.T)
GENERATED.append(save_table(fc_df, "flow_construction_audit.csv"))
mark_missing("4","Clean-loader 5-tuple keying & active timeout not explicit; merge/builder provenance unconfirmed","inspect src/clean_pipeline/{iscx_loader,vnat_loader,usbvpn_parser}.py loader internals")


,0,1,2
dataset,iscx,vnat,usbvpn
loader_file,src\clean_pipeline\iscx_loader.py,src\clean_pipeline\vnat_loader.py,src\clean_pipeline\usbvpn_parser.py
source_format,pcap/pcapng (raw),pcap (raw),"pcap (raw, per-protocol folders)"
loader_status,KNOWN,KNOWN,KNOWN
flows_native_or_reconstructed,dataset-native segmentation via clean_pipeline...,dataset-native segmentation via clean_pipeline...,dataset-native segmentation via clean_pipeline...
bidirectional,yes (directions array 0/1) [KNOWN from extractor],yes (directions array 0/1) [KNOWN from extractor],yes (directions array 0/1) [KNOWN from extractor]
direction_representation,"directions in {0,1}; fwd=1/bwd=0 [KNOWN extrac...","directions in {0,1}; fwd=1/bwd=0 [KNOWN extrac...","directions in {0,1}; fwd=1/bwd=0 [KNOWN extrac..."
packet_size_representation,absolute bytes; np.abs() applied [KNOWN extrac...,absolute bytes; np.abs() applied [KNOWN extrac...,absolute bytes; np.abs() applied [KNOWN extrac...
signed_to_abs,yes (np.abs) [KNOWN],yes (np.abs) [KNOWN],yes (np.abs) [KNOWN]
timestamps_sorted,yes (np.argsort) [KNOWN extractor L100],yes (np.argsort) [KNOWN extractor L100],yes (np.argsort) [KNOWN extractor L100]


  saved table: paper_audit_outputs\tables\flow_construction_audit.csv  (3x18)
  [MISSING/4] Clean-loader 5-tuple keying & active timeout not explicit; merge/builder provenance unconfirmed  -> needs: inspect src/clean_pipeline/{iscx_loader,vnat_loader,usbvpn_parser}.py loader internals


### 4b. Known / assumption / unknown note

In [12]:

note = """# Flow construction — known facts, assumptions, unknowns

## KNOWN (from code)
- Packet sizes use absolute value: `sz = np.abs(sizes[:n])` (feature_extractor.py L96).
- Timestamps sorted before IAT: `np.argsort(ts)` (L100) and `_iat` re-sorts (L62).
- Negative IAT clamped to 0 (clock drift): `np.maximum(diff, 0.0)` (L64).
- Packet window = first `max_packets=300` packets; `min_packets=3` drop (config/run_metadata).
- Directions are binary {0,1}; fwd=1 / bwd=0 (L106-107).
- Epsilon constant `_EPS = 1e-9` in all rate/CV denominators (L28).
- Legacy reconstructor `src/flow/builder.py`: canonical 5-tuple `A=min(endpoint),B=max(endpoint)`,
  inactivity_timeout=120s, FIN/RST close for TCP.

## ASSUMPTION
- Clean-pipeline loaders consume dataset-native pre-segmented flows (no extra re-timeout),
  i.e. ISCX/VNAT/USBVPN flows are taken as provided then windowed/feature-extracted.

## UNKNOWN / NEEDS MANUAL INPUT
- Exact 5-tuple keying inside each clean loader (iscx_loader/vnat_loader/usbvpn_parser).
- Whether `src/flow/builder.py` (legacy) actually produced the current `data/processed/*`.
- Active timeout for clean loaders (only legacy inactivity timeout is documented).
- Per-direction merge policy for USBVPN protocol subfolders.
"""
p = LOG/"flow_construction_known_assumptions_unknowns.md"
p.write_text(note, encoding="utf-8"); GENERATED.append(p)
print(note)


# Flow construction — known facts, assumptions, unknowns

## KNOWN (from code)
- Packet sizes use absolute value: `sz = np.abs(sizes[:n])` (feature_extractor.py L96).
- Timestamps sorted before IAT: `np.argsort(ts)` (L100) and `_iat` re-sorts (L62).
- Negative IAT clamped to 0 (clock drift): `np.maximum(diff, 0.0)` (L64).
- Packet window = first `max_packets=300` packets; `min_packets=3` drop (config/run_metadata).
- Directions are binary {0,1}; fwd=1 / bwd=0 (L106-107).
- Epsilon constant `_EPS = 1e-9` in all rate/CV denominators (L28).
- Legacy reconstructor `src/flow/builder.py`: canonical 5-tuple `A=min(endpoint),B=max(endpoint)`,
  inactivity_timeout=120s, FIN/RST close for TCP.

## ASSUMPTION
- Clean-pipeline loaders consume dataset-native pre-segmented flows (no extra re-timeout),
  i.e. ISCX/VNAT/USBVPN flows are taken as provided then windowed/feature-extracted.

## UNKNOWN / NEEDS MANUAL INPUT
- Exact 5-tuple keying inside each clean loader (iscx_loader/vnat_loader/usbvpn_par

## 5. Split protocol audit

Within-dataset split = per-dataset capture lists (`data/splits/{iscx,vnat}_*_captures.txt`)
and the USBVPN `split` column. Unified model split = `data/splits/clean_*_captures.txt`
(+ `clean_split_manifest.json`). LODO = hold out one dataset, train on the other two.


In [13]:

# --- 5a. Build capture->split maps per dataset ---
SPLITS_DIR = ROOT/"data"/"splits"
def read_caps(fp):
    return set(x.strip() for x in fp.read_text().splitlines() if x.strip()) if fp.exists() else None

cap_split = {}  # dataset -> {capture_id: split}
for ds in ["iscx","vnat"]:
    m = {}
    ok = True
    for sp in ["train","val","test"]:
        fp = SPLITS_DIR/f"{ds}_{sp}_captures.txt"
        caps = read_caps(fp)
        if caps is None:
            mark_missing("5", f"{ds}: split list {fp.name} missing", str(fp.relative_to(ROOT))); ok=False; continue
        for c in caps: m[c]=sp
    if ok: cap_split[ds]=m
# usbvpn: split column in processed flows
if "usbvpn" in flows and "split" in flows["usbvpn"].columns:
    u = flows["usbvpn"]
    # verify capture-level (each capture single split)
    chk = u.groupby("capture_id")["split"].nunique()
    if (chk>1).any():
        mark_missing("5","usbvpn: some captures span multiple splits (flow-level split?)","data/processed/usbvpn/flows.parquet split column")
    cap_split["usbvpn"] = u.groupby("capture_id")["split"].first().to_dict()
else:
    mark_missing("5","usbvpn: no split list and no split column","data/splits/usbvpn_*_captures.txt or split col")
print("Datasets with split maps:", list(cap_split.keys()))


Datasets with split maps: ['iscx', 'vnat', 'usbvpn']


In [14]:

# --- 5b. Within-dataset split summary + leakage check ---
split_rows, leak_rows = [], []
for ds, df in flows.items():
    if ds not in cap_split: 
        continue
    df = df.copy()
    df["split"] = df["capture_id"].map(cap_split[ds])
    unmapped = int(df["split"].isna().sum())
    if unmapped:
        mark_missing("5", f"{ds}: {unmapped} flows have capture_id not in any split list", "split list completeness")
    for sp in ["train","val","test"]:
        sub = df[df["split"]==sp]
        if len(sub)==0: 
            split_rows.append({"dataset":ds,"split":sp,"flows":0,"captures":0,"vpn_flows":0,"nonvpn_flows":0,
                               "vpn_captures":0,"nonvpn_captures":0,"both_classes":False}); continue
        lab=sub["label"].astype(int); caps=sub.groupby("capture_id")["label"].agg(["sum","count"])
        split_rows.append({
            "dataset":ds,"split":sp,"flows":len(sub),"captures":int(sub["capture_id"].nunique()),
            "vpn_flows":int((lab==1).sum()),"nonvpn_flows":int((lab==0).sum()),
            "vpn_captures":int((caps["sum"]>0).sum()),
            "nonvpn_captures":int((caps["count"]-caps["sum"]>0).sum()),
            "both_classes": bool((lab==1).any() and (lab==0).any()),
        })
    # leakage: capture overlap between splits
    sets = {sp:set(df[df["split"]==sp]["capture_id"]) for sp in ["train","val","test"]}
    leak_rows.append({"dataset":ds,
        "train_val_overlap": len(sets["train"]&sets["val"]),
        "train_test_overlap": len(sets["train"]&sets["test"]),
        "val_test_overlap": len(sets["val"]&sets["test"])})
split_df = pd.DataFrame(split_rows); leak_df = pd.DataFrame(leak_rows)
display(split_df); display(leak_df)
GENERATED.append(save_table(split_df,"within_dataset_split_summary.csv"))
GENERATED.append(save_table(leak_df,"within_dataset_leakage_check.csv"))
LEAK_OK = bool((leak_df[["train_val_overlap","train_test_overlap","val_test_overlap"]].values==0).all()) if not leak_df.empty else False
print("Within-dataset capture overlap all zero:", LEAK_OK)
if not LEAK_OK:
    print("  !! LEAKAGE DETECTED — see within_dataset_leakage_check.csv")


,dataset,split,flows,captures,vpn_flows,nonvpn_flows,vpn_captures,nonvpn_captures,both_classes
0,iscx,train,53150,104,16572,36578,28,76,True
1,iscx,val,8459,17,348,8111,1,16,True
2,iscx,test,15078,19,5513,9565,2,17,True
3,usbvpn,train,43694,357,8137,35557,20,357,True
4,usbvpn,val,7491,73,200,7291,1,73,True
5,usbvpn,test,7792,74,449,7343,2,74,True
6,vnat,train,32655,115,240,32415,56,59,True
7,vnat,val,317,24,12,305,12,12,True
8,vnat,test,739,26,127,612,14,12,True


,dataset,train_val_overlap,train_test_overlap,val_test_overlap
0,iscx,0,0,0
1,usbvpn,0,0,0
2,vnat,0,0,0


  saved table: paper_audit_outputs\tables\within_dataset_split_summary.csv  (9x9)
  saved table: paper_audit_outputs\tables\within_dataset_leakage_check.csv  (3x4)
Within-dataset capture overlap all zero: True


In [15]:

# --- 5c. Split ratios / seed / level from manifests ---
prov_rows=[]
csm = ROOT/"data"/"splits"/"clean_split_manifest.json"
if csm.exists():
    cs = json.loads(csm.read_text()); cfg = cs.get("config",{})
    prov_rows.append({"scope":"unified_clean","seed":cfg.get("seed","MISSING"),
        "train_ratio":cfg.get("train_ratio","MISSING"),"val_ratio":cfg.get("val_ratio","MISSING"),
        "test_ratio":cfg.get("test_ratio","MISSING"),"level":"capture-level (capture lists)",
        "stratified":"class-presence enforced (require_class_presence_in_val_test)"})
else:
    mark_missing("5","clean_split_manifest.json not found (ratios/seed)","data/splits/clean_split_manifest.json")
for ds in ["vnat"]:
    p = ROOT/"data"/"splits"/f"{ds}_split_manifest.json"
    if p.exists():
        d=json.loads(p.read_text()); c=d.get("config",d)
        prov_rows.append({"scope":ds,"seed":c.get("seed","?"),"train_ratio":c.get("train_ratio","?"),
            "val_ratio":c.get("val_ratio","?"),"test_ratio":c.get("test_ratio","?"),
            "level":"capture-level","stratified":"?"})
prov_df=pd.DataFrame(prov_rows); display(prov_df)
if not prov_df.empty: GENERATED.append(save_table(prov_df,"split_provenance.csv"))


,scope,seed,train_ratio,val_ratio,test_ratio,level,stratified
0,unified_clean,42,0.7,0.15,0.15,capture-level (capture lists),class-presence enforced (require_class_presenc...
1,vnat,42,?,?,?,capture-level,?


  saved table: paper_audit_outputs\tables\split_provenance.csv  (2x7)


In [16]:

# --- 5d. Leave-One-Dataset-Out (LODO) composition ---
lodo_rows=[]
all_ds=[d for d in flows if "label" in flows[d]]
for target in all_ds:
    train_ds=[d for d in all_ds if d!=target]
    tr = pd.concat([flows[d] for d in train_ds], ignore_index=True)
    te = flows[target]
    def cc(df):
        lab=df["label"].astype(int)
        return len(df), int((lab==1).sum()), int((lab==0).sum()), int(df["capture_id"].nunique())
    trf,trv,trn,trc = cc(tr); tef,tev,ten,tec = cc(te)
    lodo_rows.append({
        "target_dataset":target,"train_datasets":"+".join(train_ds),
        "validation_datasets":"(within-train val; LODO uses source val) ","test_dataset":target,
        "train_flows":trf,"test_flows":tef,
        "train_captures":trc,"test_captures":tec,
        "train_vpn":trv,"train_nonvpn":trn,"test_vpn":tev,"test_nonvpn":ten,
        "test_both_classes": bool(tev>0 and ten>0),
    })
lodo_df=pd.DataFrame(lodo_rows); display(lodo_df)
GENERATED.append(save_table(lodo_df,"lodo_composition.csv"))
mark_missing("5","LODO validation-dataset split is derived per-experiment (source train/val), not a stored file","training scripts' LODO routines (e.g. scripts/optuna_corrected_usbvpn_sensitivity.py)")


,target_dataset,train_datasets,validation_datasets,test_dataset,train_flows,test_flows,train_captures,test_captures,train_vpn,train_nonvpn,test_vpn,test_nonvpn,test_both_classes
0,iscx,usbvpn+vnat,(within-train val; LODO uses source val),iscx,92688,76687,669,140,9165,83523,22433,54254,True
1,usbvpn,iscx+vnat,(within-train val; LODO uses source val),usbvpn,110398,58977,305,504,22812,87586,8786,50191,True
2,vnat,iscx+usbvpn,(within-train val; LODO uses source val),vnat,135664,33711,644,165,31219,104445,379,33332,True


  saved table: paper_audit_outputs\tables\lodo_composition.csv  (3x13)
  [MISSING/5] LODO validation-dataset split is derived per-experiment (source train/val), not a stored file  -> needs: training scripts' LODO routines (e.g. scripts/optuna_corrected_usbvpn_sensitivity.py)


## 6. 21-feature diagnostic representation audit (`safe_core_plus_temporal`)

The 21 features are **not stored** in any parquet; they are reconstructed from raw packet
arrays via `src/clean_pipeline/feature_extractor.extract_features_batch(..., family="safe_core_plus_temporal")`.
Raw packet arrays (`timestamps`,`sizes`,`directions`) exist for **ISCX** and **VNAT** processed
flows but **not** for USBVPN (its processed table is pre-aggregated) → USBVPN is marked MISSING.


In [17]:

# --- 6a. Feature definition table (from code) ---
FEATS21 = ["total_packets","total_bytes","mean_pkt_len","std_pkt_len","median_pkt_len",
           "p25_pkt_len","p75_pkt_len","max_pkt_len","min_pkt_len","pkt_len_cv","pkt_len_iqr",
           "iat_mean","iat_std","iat_median","iat_p25","iat_p75","iat_iqr","iat_cv",
           "flow_duration","packet_rate","byte_rate"]
defs = {
 "total_packets":"n = packet count in window","total_bytes":"sum(abs(sizes))",
 "mean_pkt_len":"mean(abs(sizes))","std_pkt_len":"std(abs(sizes), ddof=0)",
 "median_pkt_len":"percentile(abs(sizes),50)","p25_pkt_len":"percentile(.,25)","p75_pkt_len":"percentile(.,75)",
 "max_pkt_len":"max(abs(sizes))","min_pkt_len":"min(abs(sizes))",
 "pkt_len_cv":"std/ max(mean,1e-9)","pkt_len_iqr":"p75-p25",
 "iat_mean":"mean(IAT)","iat_std":"std(IAT)","iat_median":"median(IAT)","iat_p25":"p25(IAT)","iat_p75":"p75(IAT)",
 "iat_iqr":"iat_p75-iat_p25","iat_cv":"iat_std/ max(iat_mean,1e-9)",
 "flow_duration":"ts[-1]-ts[0] (0 if n<2)","packet_rate":"n / max(duration,1e-9)","byte_rate":"sum(sizes)/max(duration,1e-9)",
}
defs_df = pd.DataFrame([{"feature":f,"definition":defs[f]} for f in FEATS21])
GENERATED.append(save_table(defs_df,"feature_definitions_21.csv"))
display(defs_df)


  saved table: paper_audit_outputs\tables\feature_definitions_21.csv  (21x2)


,feature,definition
0,total_packets,n = packet count in window
1,total_bytes,sum(abs(sizes))
2,mean_pkt_len,mean(abs(sizes))
3,std_pkt_len,"std(abs(sizes), ddof=0)"
4,median_pkt_len,"percentile(abs(sizes),50)"
5,p25_pkt_len,"percentile(.,25)"
6,p75_pkt_len,"percentile(.,75)"
7,max_pkt_len,max(abs(sizes))
8,min_pkt_len,min(abs(sizes))
9,pkt_len_cv,"std/ max(mean,1e-9)"


In [18]:

# --- 6b. Reconstruct 21 features for datasets with raw packet arrays ---
# Preferred: the project's extractor. Several files in src/clean_pipeline are CORRUPTED
# (all-null-bytes: __init__.py, config.py, run_pipeline.py, splitter.py), which breaks the
# package import. We try the real import; on failure we fall back to an INLINE faithful
# replica of src/clean_pipeline/feature_extractor.extract_flow_features (formulas verified
# against the source, L71-150), so the reconstruction still uses identical formulas.
EXTRACT_SOURCE = None
extract_flow_features = None
try:
    from src.clean_pipeline.feature_extractor import extract_flow_features as _eff
    extract_flow_features = _eff; EXTRACT_SOURCE = "project import (src.clean_pipeline.feature_extractor)"
except Exception as e:
    mark_missing("6", f"project extractor import failed ({e.__class__.__name__}: {e})",
                 "repair corrupted all-null files in src/clean_pipeline/ (__init__.py, config.py, run_pipeline.py, splitter.py)")
    _EPS = 1e-9
    def _safe_stats(a):
        a = np.asarray(a, dtype=np.float64)
        if a.size == 0:
            return dict(count=0.,sum=0.,mean=0.,std=0.,min=0.,p25=0.,median=0.,p75=0.,max=0.)
        return dict(count=float(a.size),sum=float(a.sum()),mean=float(a.mean()),std=float(a.std(ddof=0)),
                    min=float(a.min()),p25=float(np.percentile(a,25)),median=float(np.percentile(a,50)),
                    p75=float(np.percentile(a,75)),max=float(a.max()))
    def _iat(ts):
        ts = np.asarray(ts, dtype=np.float64)
        if ts.size <= 1: return np.array([], dtype=np.float64)
        return np.maximum(np.diff(np.sort(ts)), 0.0)
    def extract_flow_features(timestamps, sizes, directions, *, max_packets=300):
        n = min(len(timestamps), len(sizes), len(directions), max_packets)
        ts = np.asarray(timestamps[:n], dtype=np.float64)
        sz = np.abs(np.asarray(sizes[:n], dtype=np.float64))
        order = np.argsort(ts); ts = ts[order]; sz = sz[order]
        ia = _iat(ts); ist = _safe_stats(ia); ss = _safe_stats(sz)
        dur = float(ts[-1]-ts[0]) if n >= 2 else 0.0
        f = {}
        f["total_packets"]=float(n); f["total_bytes"]=ss["sum"]; f["mean_pkt_len"]=ss["mean"]
        f["std_pkt_len"]=ss["std"]; f["median_pkt_len"]=ss["median"]; f["p25_pkt_len"]=ss["p25"]
        f["p75_pkt_len"]=ss["p75"]; f["max_pkt_len"]=ss["max"]; f["min_pkt_len"]=ss["min"]
        f["iat_mean"]=ist["mean"]; f["iat_std"]=ist["std"]; f["iat_median"]=ist["median"]
        f["iat_p25"]=ist["p25"]; f["iat_p75"]=ist["p75"]
        f["flow_duration"]=dur; f["packet_rate"]=float(n)/max(dur,_EPS); f["byte_rate"]=ss["sum"]/max(dur,_EPS)
        f["iat_cv"]=ist["std"]/max(ist["mean"],_EPS); f["iat_iqr"]=ist["p75"]-ist["p25"]
        f["pkt_len_cv"]=ss["std"]/max(ss["mean"],_EPS); f["pkt_len_iqr"]=ss["p75"]-ss["p25"]
        return f
    EXTRACT_SOURCE = "INLINE faithful replica (project package corrupted)"
print("Extractor source:", EXTRACT_SOURCE)

def build_feat21(df, ds, min_packets=3, max_packets=300):
    rows = []
    for r in df[["flow_id","capture_id","label","timestamps","sizes","directions"]].itertuples(index=False):
        ts, sz, dr = r.timestamps, r.sizes, r.directions
        n = min(len(ts), len(sz), len(dr))
        if n < min_packets:   # mirrors extract_features_batch drop
            continue
        feat = extract_flow_features(np.asarray(ts,dtype=np.float64),
                                     np.asarray(sz,dtype=np.float64),
                                     np.asarray(dr,dtype=np.int32), max_packets=max_packets)
        feat.update({"flow_id":r.flow_id,"capture_id":r.capture_id,"dataset":ds,"label":int(r.label)})
        rows.append(feat)
    out = pd.DataFrame(rows)
    keep = ["flow_id","capture_id","dataset","label"] + [c for c in FEATS21 if c in out.columns]
    out = out[keep].copy()
    for c in FEATS21:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out.replace([np.inf,-np.inf], 0.0)

feat21 = {}
for ds in ["iscx","vnat"]:
    df = flows.get(ds)
    if df is None:
        continue
    need = {"timestamps","sizes","directions","flow_id","capture_id","label"}
    if not need.issubset(df.columns):
        mark_missing("6", f"{ds}: raw packet arrays missing {need-set(df.columns)}", f"data/processed/{ds}/flows.parquet")
        continue
    out = build_feat21(df, ds)
    feat21[ds] = out
    print(f"{ds}: reconstructed {out.shape}  (min_packets=3 drop applied)")
# usbvpn explicitly missing
if "usbvpn" in flows and not {"timestamps","sizes","directions"}.issubset(flows["usbvpn"].columns):
    mark_missing("6","usbvpn: 21-feature reconstruction impossible (processed table is pre-aggregated; no packet arrays)",
                 "raw USBVPN packet arrays / re-run clean pipeline persisting safe_core_plus_temporal for usbvpn")


  [MISSING/6] project extractor import failed (SyntaxError: source code string cannot contain null bytes)  -> needs: repair corrupted all-null files in src/clean_pipeline/ (__init__.py, config.py, run_pipeline.py, splitter.py)
Extractor source: INLINE faithful replica (project package corrupted)


iscx: reconstructed (11801, 25)  (min_packets=3 drop applied)


vnat: reconstructed (8107, 25)  (min_packets=3 drop applied)
  [MISSING/6] usbvpn: 21-feature reconstruction impossible (processed table is pre-aggregated; no packet arrays)  -> needs: raw USBVPN packet arrays / re-run clean pipeline persisting safe_core_plus_temporal for usbvpn


In [19]:

# --- 6c. Per-feature summary by dataset (only reconstructed datasets) ---
sum_rows=[]
for ds, out in feat21.items():
    for f in FEATS21:
        if f not in out.columns:
            mark_missing("6", f"{ds}: feature {f} not produced", "feature_extractor family output"); continue
        s = pd.to_numeric(out[f], errors="coerce")
        sum_rows.append({"dataset":ds,"feature":f,"dtype":str(out[f].dtype),
            "missing":int(s.isna().sum()),"infinite":int(np.isinf(s.values).sum()),
            "min":float(np.nanmin(s)),"Q1":float(np.nanpercentile(s,25)),"median":float(np.nanpercentile(s,50)),
            "mean":float(np.nanmean(s)),"Q3":float(np.nanpercentile(s,75)),"max":float(np.nanmax(s)),
            "std":float(np.nanstd(s))})
if sum_rows:
    fs_df=pd.DataFrame(sum_rows); display(fs_df.head(21))
    GENERATED.append(save_table(fs_df,"feature_summary_21_by_dataset.csv"))
else:
    print("No 21-feature reconstruction available; see MISSING markers.")


,dataset,feature,dtype,missing,infinite,min,Q1,median,mean,Q3,max,std
0,iscx,total_packets,float64,0,0,3.000000e+00,5.000000,8.000000,23.975680,28.000000,1.000000e+02,3.063388e+01
1,iscx,total_bytes,float64,0,0,1.250000e+02,656.000000,1200.000000,10751.859588,6001.000000,5.556030e+05,3.122460e+04
2,iscx,mean_pkt_len,float64,0,0,3.000000e+01,80.066667,150.000000,248.299435,256.734940,5.556030e+03,3.261548e+02
3,iscx,std_pkt_len,float64,0,0,0.000000e+00,0.000000,50.263438,226.957076,357.190075,9.407665e+03,4.787466e+02
4,iscx,median_pkt_len,float64,0,0,3.000000e+01,66.000000,84.000000,162.942166,150.000000,2.962000e+03,2.741755e+02
5,iscx,p25_pkt_len,float64,0,0,3.000000e+01,54.000000,66.000000,89.409457,85.000000,1.474000e+03,9.072846e+01
6,iscx,p75_pkt_len,float64,0,0,3.000000e+01,81.750000,150.000000,332.625985,279.500000,1.110450e+04,5.776976e+02
7,iscx,max_pkt_len,float64,0,0,3.000000e+01,118.000000,204.000000,825.374799,1360.000000,6.429400e+04,2.096444e+03
8,iscx,min_pkt_len,float64,0,0,3.000000e+01,54.000000,66.000000,79.298449,79.000000,1.404000e+03,4.726975e+01
9,iscx,pkt_len_cv,float64,0,0,0.000000e+00,0.000000,0.423581,0.605137,1.103759,6.142338e+00,5.926876e-01


  saved table: paper_audit_outputs\tables\feature_summary_21_by_dataset.csv  (42x12)


In [20]:

# --- 6d. Correlation heatmaps + key-feature distributions ---
for ds, out in feat21.items():
    cols=[f for f in FEATS21 if f in out.columns]
    if len(cols)<3: continue
    corr = out[cols].corr()
    fig, ax = plt.subplots(figsize=(9,7))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90, fontsize=7)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title(f"21-feature correlation — {ds}")
    GENERATED.append(save_fig(fig, f"corr_heatmap_21_{ds}.png"))
# distributions for important features
important=["total_packets","mean_pkt_len","flow_duration","packet_rate","iat_cv","pkt_len_cv"]
for ds, out in feat21.items():
    avail=[f for f in important if f in out.columns]
    if not avail: continue
    fig, axes = plt.subplots(2,3, figsize=(13,7)); axes=axes.ravel()
    for i,f in enumerate(avail):
        v=pd.to_numeric(out[f],errors="coerce").replace([np.inf,-np.inf],np.nan).dropna()
        v=v[v.between(v.quantile(0.01), v.quantile(0.99))] if len(v)>10 else v
        axes[i].hist(v, bins=40, color="#4C72B0"); axes[i].set_title(f"{ds}: {f}", fontsize=9)
    for j in range(len(avail),6): axes[j].axis("off")
    fig.suptitle(f"Key 21-feature distributions — {ds}")
    GENERATED.append(save_fig(fig, f"feature_dist_21_{ds}.png"))


  saved figure: paper_audit_outputs\figures\corr_heatmap_21_iscx.png


  saved figure: paper_audit_outputs\figures\corr_heatmap_21_vnat.png


  saved figure: paper_audit_outputs\figures\feature_dist_21_iscx.png


  saved figure: paper_audit_outputs\figures\feature_dist_21_vnat.png


## 7. Final summary

In [21]:

# --- 7. Final summary + checklist ---
total_datasets = len([d for d in flows if 'label' in flows[d]])
total_flows = int(sum(len(flows[d]) for d in flows))
total_captures = int(sum(flows[d]['capture_id'].nunique() for d in flows if 'capture_id' in flows[d]))
split_complete = bool(len(cap_split)==total_datasets) and ("usbvpn" in cap_split)
missing_preproc = [m for m in MISSING if m["section"]=="3"]
missing_flowcon = [m for m in MISSING if m["section"]=="4"]

print("="*60)
print("AUDIT FINAL SUMMARY")
print("="*60)
print(f"Datasets audited      : {total_datasets}  ({sorted([d for d in flows])})")
print(f"Total flows           : {total_flows}")
print(f"Total captures        : {total_captures}")
print(f"Split details complete: {split_complete}")
print(f"Leakage checks passed  : {LEAK_OK}")
print(f"MISSING items total    : {len(MISSING)}  (preproc={len(missing_preproc)}, flow-construction={len(missing_flowcon)})")
print(f"Files generated        : {len(GENERATED)}")

save_metric({
    "datasets_audited": total_datasets,
    "total_flows": total_flows, "total_captures": total_captures,
    "split_details_complete": split_complete, "leakage_checks_passed": LEAK_OK,
    "n_missing_items": len(MISSING), "missing_items": MISSING,
    "n_files_generated": len(GENERATED),
    "files_generated": [str(Path(p).relative_to(ROOT)) for p in GENERATED],
}, "audit_final_summary.json")
save_table(pd.DataFrame(MISSING) if MISSING else pd.DataFrame([{"section":"-","what":"none","needed":"-"}]),
           "audit_missing_items.csv")

lines = ["# 01 — Data Preprocessing & Split Audit — Checklist\n",
    f"_Generated: {datetime.now().isoformat(timespec='seconds')}_\n",
    f"- Datasets audited: **{total_datasets}** ({', '.join(sorted(flows))})",
    f"- Total flows: **{total_flows}**",
    f"- Total captures: **{total_captures}**",
    f"- Split details complete: **{split_complete}**",
    f"- Leakage checks passed (within-dataset capture overlap == 0): **{LEAK_OK}**",
    f"- MISSING / NEEDS MANUAL INPUT items: **{len(MISSING)}**",
    "",
    "## Section status",
    "- [x] 1. Environment & file discovery",
    "- [x] 2. Dataset composition",
    "- [x] 3. Cleaning/filtering (partial — see MISSING for un-logged per-flow counters)",
    "- [x] 4. Flow-construction & loader audit (code-derived; some loader internals UNKNOWN)",
    "- [x] 5. Split protocol & leakage check",
    "- [{}] 6. 21-feature reconstruction (ISCX/VNAT computed; USBVPN MISSING — no packet arrays)".format("x" if feat21 else " "),
    "- [x] 7. Final summary",
    "",
    "## MISSING / NEEDS MANUAL INPUT",
]
if MISSING:
    for m in MISSING:
        lines.append(f"- **[S{m['section']}]** {m['what']} → _needs:_ {m['needed']}")
else:
    lines.append("- none")
lines += ["", "## Files generated"]
for p in GENERATED:
    lines.append(f"- `{Path(p).relative_to(ROOT)}`")
chk = LOG/"01_data_preprocessing_split_checklist.md"
chk.write_text("\n".join(lines), encoding="utf-8")
print("\nChecklist saved:", chk.relative_to(ROOT))


AUDIT FINAL SUMMARY
Datasets audited      : 3  (['iscx', 'usbvpn', 'vnat'])
Total flows           : 169375
Total captures        : 809
Split details complete: True
Leakage checks passed  : True
MISSING items total    : 10  (preproc=6, flow-construction=1)
Files generated        : 23
  saved metric: paper_audit_outputs\metrics\audit_final_summary.json
  saved table: paper_audit_outputs\tables\audit_missing_items.csv  (10x3)

Checklist saved: paper_audit_outputs\logs\01_data_preprocessing_split_checklist.md
